In [0]:
from pyspark.sql.functions import sum, avg, round

# monthly sales trend
monthly_sales = spark.read.table("ecommerce.e_comm_gold.factSales")
monthly_sales.createOrReplaceTempView("product_revenue_summary")

# calculate metrics 
agg_product_sales = spark.sql("""        
        select 
            product_key
            , count(product_key) as products_sold
            , sum(order_quantity) as total_products_purchased
            , round(sum(sale_amount),2) as total_revenue
            , round(avg(sale_amount),2) as avg_order_value
            , round(avg(order_quantity),2) as avg_items_sold_per_product
            , current_timestamp() as load_ts
        from product_revenue_summary
        where dq_note = "is_valid"
        group by product_key
        """)
    
# write to delta table

agg_product_sales.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.fact_agg_product_revenue_summary")